In [ ]:
import os

project_dir = os.getcwd()[:os.getcwd().find('SERPENS')+7]
os.chdir(project_dir)
print("Current Working Directory:", os.getcwd())

In [ ]:
from src.serpens_simulation import SerpensSimulation
from src.serpens_analyzer import SerpensAnalyzer
try:
    from src.cerpens import CerpensSimulation
except (OSError, ImportError, ModuleNotFoundError):
    pass
from src.species import Species
import matplotlib

### Setting up a Simulation for Testing and Visualization Examples

In [ ]:
from src.parameters import GLOBAL_PARAMETERS
GLOBAL_PARAMETERS.reset()

In [ ]:
from src.grains import GrainPopulation

# Grain population
LAUNCH_SPEED_M_S = 30000.

grain_pop = GrainPopulation(
    name='synthetic size distribution',
    density_kg_m3=2000.0,
    radius_min_m=0.3e-6, radius_max_m=3e-6,
    size_slope=3.5,
    tracers_per_injection=64,
    dust_mass_per_sec=1.0,
    speed_ref_m_s=LAUNCH_SPEED_M_S,
    radius_ref_m=1e-6,
    speed_exponent=0.5, speed_min_m_s=0.5 * LAUNCH_SPEED_M_S, speed_max_m_s=2 * LAUNCH_SPEED_M_S,
    launch_altitude_m=1e4,
    radiation_source='star',
    luminosity_w=0.58 * 3.828e26,
    q_pr=1.0,
    potential_v=0.0,
)

In [ ]:
# Create a simulation with the Jupiter system
sim = SerpensSimulation(system="55-Cnc-e")

# Declare Io as a source for Sodium
sim.object_to_source(
    "planet",
    species=[
        Species(
            'SiO2',
            n_th=0, n_sp=200,
            mass_per_sec=4.892e6,
            sput_spec=dict(
                v_b=23_000, #0.95*1000*10,
                v_M=40_000, #15.24*10000*10,
            ),
            lifetime=2.2*3600,
            beta=5e-4,
            charge_e=0,
        ),
    ],
    grains=grain_pop
)

# Run the simulation for x orbits with y spawns
sim.advance(orbits=4, spawns=36, orbits_reference="planet", verbose=True)

### Creating an Analyzer

Now that we have simulation data, let's create an analyzer to visualize it.

In [ ]:
# Create an analyzer with Io as the reference system
sa = SerpensAnalyzer(reference_system="planet")

### Phase Curve Calculation and Visualization

The `calculate_phasecurve` function calculates the phase curve data, and `plot_phasecurve` returns a plotly figure to visualize it.

In [ ]:
# Calculate the phase curve
sa.calculate_phasecurve('planet', orbits=3)     # More than 1 orbit needs to be run. Otherwise errors may occur.

In [ ]:
# Plot the phase curve
fig = sa.plot_phasecurve(
    filename='phase-curve.csv',
    column_density=True,    # Show column density
    particle_density=True,  # Show particle density
    type="mean"              # Use maximum density values
)
fig.show()

### Web dashboard with an existing analyzer

These optional cells are intentionally unexecuted. Install from the repository with `python -m pip install -e '.[web]'` (Python >=3.9, Dash >=2.11,<4 with native Jupyter support). They expect an existing analyzer named `analyzer`; if using the legacy examples above, bind `analyzer = sa` first. Do not rerun simulations or copy outputs for these examples. Keep the working directory at the trusted dataset directory containing its existing `simdata/`.

The recommended helper defaults to `jupyter_mode='external'`; pass `jupyter_mode='inline'` for embedding. It caches the app on the analyzer and reuses the server on port 8050 on repeated invocation. A different port is rejected to prevent duplicate servers. Use one dataset per kernel: changing `cwd`/`GLOBAL_PARAMETERS` or concurrently mutating the analyzer outside the app is unsupported.

See the [visualization guide](../docs/visualization.md) for planar, LOS, 3D, profile, and phase views, explicit phase computation or local CSV loading, CSV/HTML downloads and modebar images (no Kaleido), refresh, and geometry limitations. Grain fields use SI; grain LOS is projected 3D density, not an integrated column, and geometric area is not optical depth. Legacy APIs remain available.

In [ ]:
from src.visualizing.webapp import launch_notebook

app = launch_notebook(sa, port=8051)

### Shared scientific provider and explicit figure display

This alternative needs no web server or Dash. Use an existing timestep (index 1 requires at least two snapshots). `species_ids=None` selects all gas species. The inclusive log10 threshold for species ID 1 hides values below 2 without recomputing density; `None` clears a bound. Rebuild with `DisplayOptions()` to reset thresholds using the same prepared data. The caller explicitly invokes `.show()`; building a figure does not display or save it. Keep `provider` for scientific-cache reuse and call `provider.refresh()` after on-disk data change. Run this separately from active dashboard computations, not concurrently against the same analyzer.

In [ ]:
from src.visualizing.data import DisplayOptions, PlotRequest, VisualizationDataProvider
from src.visualizing.figures import build_figure

provider = VisualizationDataProvider(sa)
request = PlotRequest(timestep=12, view='planar', species_ids=None, dimension=3)
data = provider.prepare(request)
fig = build_figure(data, DisplayOptions(thresholds={'1': (2, None)}))
fig.show()